# H2 · "Competition destroys profits" (*Zero to One*, ch. 3–4)

> "Under perfect competition, in the long run no company makes an economic profit." … "Monopoly is the condition of every successful business."

Thiel's example: in 2012 US airlines earned about 37 cents per passenger trip, while Google's profit margin was about 21%.

| Claim | Test | Data |
|---|---|---|
| **H2a.** Concentrated industries earn higher margins. | Price-cost margin against the HHI and the share of the 4 largest firms (CR4), across about 350 six-digit manufacturing industries, in 2017 and 2022. Also within industries: did a rise in concentration from 2017 to 2022 raise margins? | Census Economic Census (concentration), Annual Survey of Manufactures (margins), using your Census key |
| **H2b.** Within an industry, the firm with the larger share earns the higher margin. | Firm operating margin against revenue share within its 4-digit SIC industry, 2011–2025, comparing firms in the same industry and year, with standard errors clustered by firm. Pre- and post-2014 slopes are compared, and a separate test asks whether dominating a niche pays at a given firm size. | SEC XBRL annual frames; SIC codes from SEC Financial Statement Data Sets |
| **H2c.** Thiel's own example. | Operating margins of US scheduled airlines (SIC 4512) against the large tech firms, 2011–2025. | SEC XBRL |

**What would count against Thiel.** No relationship, or a negative one, between concentration and margins. Or a positive cross-sectional link that disappears within industries, which would mean concentration and margins share a cause (for example, scale economies) rather than one producing the other.

**Caveats**
* A price-cost margin is an accounting margin, not an economic profit: it ignores the cost of capital. Capital-intensive industries can show high margins without earning monopoly rents.
* National concentration ratios ignore imports and regional markets.
* Public-firm "market share" uses only listed rivals, so it overstates concentration in industries with many private firms.
* The Census concentration years (2017, 2022) both fall after the book. The 2017→2022 change tests the claim on post-publication data; the "before" comparison comes from H2b.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import thiel_common as tc
tc.style()
print("synthetic mode" if tc.SYNTHETIC else "live data")

## Part A · Manufacturing industries: concentration and price-cost margins (Census)

The **price-cost margin** is PCM = (value added − payroll) / value of shipments. Value added already nets out materials, so PCM is the share of each sales dollar left after materials and labour (Collins & Preston, 1969). The Census API sometimes rejects a query form, so each request below tries a few variants and prints the server's message if all of them fail.

In [ ]:
def first_ok(dataset, variants, name):
    errs = []
    for v in variants:
        try:
            return tc.census_get(dataset, v, name)
        except Exception as e:
            errs.append(str(e)[:250])
    raise RuntimeError(f"All query forms failed for {dataset}:\n" + "\n".join(errs))

def concentration(year):
    nv = f"NAICS{year}"
    base = {"get": f"{nv},{nv}_LABEL,CONCENFI,CONCENFI_LABEL,VAL_PCT,HHI,RCPTOT,FIRM", "for": "us:*"}
    d = first_ok(f"{year}/ecnsize", [{**base, "SECTOR": "31-33"}, base, {**base, nv: "*"}], f"concen_{year}")
    d = d.rename(columns={nv: "naics"})
    lab = d.get("CONCENFI_LABEL", d["CONCENFI"]).astype(str)
    cr4 = d[lab.str.contains("4 largest") | (d.CONCENFI == "002")].groupby("naics").VAL_PCT.first()
    hhi = pd.to_numeric(d.HHI, errors="coerce").groupby(d.naics).max()
    out = pd.DataFrame({"cr4": pd.to_numeric(cr4, errors="coerce"), "hhi": hhi})
    out = out[out.index.astype(str).str.len() == 6]                          # 6-digit industries only
    return out.dropna(how="all")

def asm():
    base = {"get": "NAICS2017,YEAR,RCPTOT,VALADD,PAYANN,CSTMTOT", "for": "us:*"}
    frames = []
    for ds in ["timeseries/asm/benchmark2022", "timeseries/asm/benchmark2017"]:
        try:
            frames.append(first_ok(ds, [base, {**base, "time": "from 2012 to 2022"}], ds.split("/")[-1]))
        except Exception as e:
            print("skipped", ds, "->", str(e)[:200])
    d = pd.concat(frames).rename(columns={"NAICS2017": "naics"})
    for c in ["RCPTOT", "VALADD", "PAYANN", "CSTMTOT"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["year"] = pd.to_numeric(d.get("YEAR", d.get("time")), errors="coerce")
    d = d[d.naics.astype(str).str.len() == 6].drop_duplicates(["naics", "year"], keep="first")
    d["pcm"] = (d.VALADD - d.PAYANN) / d.RCPTOT
    return d[(d.RCPTOT > 0) & d.pcm.between(-0.5, 1)]

con = {y: concentration(y) for y in (2017, 2022)}
m = asm()
print({y: len(c) for y, c in con.items()}, "industries with concentration;", m.year.value_counts().sort_index().to_dict())

In [ ]:
def year_near(df, y):
    yrs = sorted(df.year.unique()); best = min(yrs, key=lambda v: abs(v - y))
    return df[df.year == best].set_index("naics"), best

rows = []
for y in (2017, 2022):
    mm, yy = year_near(m, y)
    j = con[y].join(mm[["pcm", "RCPTOT"]], how="inner").assign(census_year=y, asm_year=yy)
    rows.append(j)
    print(f"{y}: {len(j)} industries matched (ASM year {yy})")
xs = pd.concat(rows).reset_index().rename(columns={"index": "naics"})
xs["log_hhi"] = np.log(xs.hhi.clip(lower=1))
tc.save_table(xs, "h2a_industry_cross_section", index=False)

res = []
for y, g in xs.groupby("census_year"):
    for x in ["log_hhi", "cr4"]:
        gg = g.dropna(subset=[x, "pcm"])
        t = tc.fe_ols(gg.assign(c=range(len(gg))), "pcm", [x], [], "c")
        res.append({"year": y, "x": x, "coef": t.loc[x, "coef"], "se": t.loc[x, "se"], "p": t.loc[x, "p"], "n": len(gg),
                    "spearman": gg[[x, "pcm"]].corr("spearman").iloc[0, 1]})
cs = pd.DataFrame(res); tc.save_table(cs, "h2a_cross_section_slopes", index=False)
cs.round(4)

### Within industries: did a rise in concentration from 2017 to 2022 raise margins?

First differences remove everything fixed about an industry (its technology, its capital intensity). Thiel's argument predicts a positive slope.

In [ ]:
w = xs.pivot_table(index="naics", columns="census_year", values=["cr4", "log_hhi", "pcm"])
fd = pd.DataFrame({"d_cr4": w["cr4"][2022] - w["cr4"][2017], "d_log_hhi": w["log_hhi"][2022] - w["log_hhi"][2017],
                   "d_pcm": w["pcm"][2022] - w["pcm"][2017]}).dropna()
fdr = []
for x in ["d_cr4", "d_log_hhi"]:
    g = fd.dropna(subset=[x]); t = tc.fe_ols(g.assign(c=range(len(g))), "d_pcm", [x], [], "c")
    fdr.append({"x": x, "coef": t.loc[x, "coef"], "se": t.loc[x, "se"], "p": t.loc[x, "p"], "n": len(g)})
fdr = pd.DataFrame(fdr); tc.save_table(fdr, "h2a_first_differences", index=False)
print(f"{len(fd)} industries in both years; median change in CR4: {fd.d_cr4.median():+.1f} pts")
fdr.round(4)

### Figure H2-1 · Price-cost margin by concentration (manufacturing, binned)

Industries are grouped into 20 bins of equal count by HHI. Each dot shows a bin's average HHI and average margin.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, (y, g) in enumerate(xs.dropna(subset=["log_hhi", "pcm"]).groupby("census_year")):
    b = g.assign(bin=pd.qcut(g.log_hhi, 20, labels=False, duplicates="drop")).groupby("bin")[["hhi", "pcm"]].mean()
    axes[0].plot(b.hhi, b.pcm * 100, "o", color=tc.C[i], ms=7, label=f"{y} ({len(g)} industries)")
axes[0].set_xscale("log")
from matplotlib.ticker import FuncFormatter
axes[0].xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}")); axes[0].xaxis.set_minor_formatter(FuncFormatter(lambda v, _: ""))
axes[0].set_xlabel("HHI, 50 largest firms (log)"); axes[0].set_ylabel("Price-cost margin, %")
axes[0].set_title("Price-cost margin by HHI bin (20 bins)", loc="left"); axes[0].legend(loc="upper left")
ax = axes[1]
b = fd.assign(bin=pd.qcut(fd.d_cr4, 15, labels=False, duplicates="drop")).groupby("bin")[["d_cr4", "d_pcm"]].mean()
ax.scatter(fd.d_cr4, fd.d_pcm * 100, s=8, color=tc.GREY, alpha=0.4)
ax.plot(b.d_cr4, b.d_pcm * 100, "o-", color=tc.C[0], ms=6, lw=1.2, label="Binned means")
ax.axhline(0, color=tc.INK, lw=0.6); ax.axvline(0, color=tc.INK, lw=0.6)
ax.set_xlim(fd.d_cr4.quantile(0.01), fd.d_cr4.quantile(0.99)); ax.set_ylim(np.nanpercentile(fd.d_pcm * 100, [1, 99]))
ax.set_xlabel("Change in CR4, 2017 to 2022 (points)"); ax.set_ylabel("Change in price-cost margin (points)")
ax.set_title("Within-industry changes", loc="left"); ax.legend(loc="upper left")
fig.tight_layout(); tc.save_fig(fig, "h2_fig1_census_margins")

## Part B · Public firms: does market share predict margins?

* **Market share** is the firm's revenue divided by the total revenue of SEC filers in its 4-digit SIC industry that year. Industries need at least 3 filers.
* Financials (SIC 6000–6799) and public administration (9xxx) are excluded, because operating margin means little for banks and insurers.
* Firms with revenue below $10m, or operating margins outside ±100%, are dropped; the extreme margins are mostly early-stage firms or tagging errors.

First run: about 100 requests to SEC XBRL frames (15 years × 7 tags), plus 4 Financial Statement Data Set zips of about 50–90 MB each, used only for the SIC codes.

In [ ]:
fin = tc.annual_financials(range(2011, 2026))
for c in ["revenue", "op_income", "net_income", "assets"]:          # SEC values can arrive as Python ints/objects
    fin[c] = pd.to_numeric(fin[c], errors="coerce").astype(float)
fin["year"] = fin["year"].astype(int)
sic = tc.sec_sic()
fin["sic"] = fin.cik.map(sic)
fin = fin.dropna(subset=["sic", "revenue"])
s2 = fin.sic.str[:2].astype(int)
fin = fin[~s2.between(60, 67) & (s2 < 90) & (fin.revenue >= 1e7)].copy()
# Revenue tagged at the wrong scale: no US firm has $1tn of revenue (Walmart, the largest, has about $0.7tn),
# and revenue above 20x total assets for a $1bn+ firm is almost always a 1,000x tagging error (e.g. Tigo Energy).
bad = (fin.revenue >= 1e12) | ((fin.revenue > 1e9) & (fin.revenue > 20 * fin.assets))
print("dropped for revenue scale errors:", int(bad.sum()), "firm-years:", ", ".join(sorted(fin[bad].entityName.str.title().unique())[:8]))
fin = fin[~bad].copy()
fin["op_margin"] = fin.op_income / fin.revenue
fin["roa"] = (fin.net_income / fin.assets.where(fin.assets > 0)).replace([np.inf, -np.inf], np.nan)
fin = fin[fin.op_margin.between(-1, 1)]
fin.loc[~fin.roa.between(-1, 1), "roa"] = np.nan
fin["ind_year"] = fin.sic + "_" + fin.year.astype(str)
fin["sector_year"] = fin.sic.str[:2] + "_" + fin.year.astype(str)
tot = fin.groupby("ind_year").revenue.transform("sum"); nfirm = fin.groupby("ind_year").revenue.transform("size")
fin["share"] = fin.revenue / tot
fin = fin[nfirm >= 3].copy()
fin["rank"] = fin.groupby("ind_year").revenue.rank(ascending=False, method="first")
fin["leader"] = (fin["rank"] == 1).astype(float)
fin["dominant"] = (fin.share >= 0.5).astype(float)
fin["log_rev"] = np.log(fin.revenue.to_numpy(dtype=float))
fin["post"] = (fin.year >= 2015).astype(float)
fin["share_x_post"] = fin.share * fin.post
print(f"{len(fin):,} firm-years, {fin.cik.nunique():,} firms, {fin.sic.nunique()} SIC industries, {fin.year.min()}-{fin.year.max()}")
fin.groupby("year").size().to_frame("firms").T

### Regressions

Within one industry and year, market share is revenue divided by a fixed total, so share and firm size cannot be separated there. The specifications are built around that:

| Specification | Fixed effects | Question |
|---|---|---|
| share | industry × year | Within an industry, do bigger-share firms earn more? (Thiel's claim, scale included) |
| leader, dominant (≥50%) | industry × year | Does being #1, or holding half the market, pay? |
| share, pre vs post 2014 | industry × year | Has the payoff to share changed since the book? |
| share + size | broad sector (SIC 2-digit) × year | Holding absolute size fixed, does dominating a *smaller* industry pay? This is Thiel's "dominate a niche" against plain economies of scale. |

Standard errors are clustered by firm.

In [ ]:
specs = {
    "share":                     (["share"], ["ind_year"]),
    "leader":                    (["leader"], ["ind_year"]),
    "dominant (>=50%)":          (["dominant"], ["ind_year"]),
    "share, pre vs post 2014":   (["share", "share_x_post"], ["ind_year"]),
    "share + size, sector-year": (["share", "log_rev"], ["sector_year"]),
}
out = []
for dep in ["op_margin", "roa"]:
    for lab, (xs_, fes) in specs.items():
        t = tc.fe_ols(fin, dep, xs_, fes, "cik")
        for x in xs_:
            out.append({"outcome": dep, "spec": lab, "x": x, **t.loc[x, ["coef", "se", "p", "n", "clusters"]].to_dict()})
reg = pd.DataFrame(out)
for c in ["coef", "se", "p", "n", "clusters"]:
    reg[c] = pd.to_numeric(reg[c], errors="coerce")
tc.save_table(reg, "h2b_firm_regressions", index=False)
reg.round(4)

### Figure H2-2 · Operating margin by market-share bin, before and after the book

Firms are binned by market share within their industry and year. Each line plots the median operating margin per bin, after removing industry-year means. Removing those means stops low-margin and high-margin industries from driving the pattern.

In [ ]:
fin["margin_dm"] = fin.op_margin - fin.groupby("ind_year").op_margin.transform("mean")
bins = [0, .01, .02, .05, .1, .2, .35, .5, .75, 1.0001]
labels = ["<1%", "1-2%", "2-5%", "5-10%", "10-20%", "20-35%", "35-50%", "50-75%", ">75%"]
fin["share_bin"] = pd.cut(fin.share, bins, labels=labels, right=False)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for i, (lab, g) in enumerate([("2011-2014", fin[fin.year <= 2014]), ("2015-2025", fin[fin.year >= 2015])]):
    s = g.groupby("share_bin", observed=True).margin_dm.median() * 100
    n = g.groupby("share_bin", observed=True).size()
    ax.plot(range(len(s)), s.values, "o-", color=tc.C[i], ms=6, label=f"{lab} ({len(g):,} firm-years)")
ax.axhline(0, color=tc.INK, lw=0.6)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_xlabel("Revenue share of the firm's 4-digit SIC industry (public filers)")
ax.set_ylabel("Operating margin minus industry-year mean, pts (median)")
ax.set_title("Operating margin by market-share band", loc="left"); ax.legend(loc="upper left")
fig.tight_layout(); tc.save_fig(fig, "h2_fig2_share_margin")

## Part C · Thiel's example, updated: airlines against the big tech firms

Airlines are every filer with SIC 4512 (scheduled air transport). "Big tech" is Alphabet (Google), Apple, Microsoft, Meta and Amazon, identified by SEC CIK. Margins are aggregate: the sum of operating income divided by the sum of revenue.

In [ ]:
BIGTECH = {1652044: "Alphabet", 1288776: "Alphabet", 320193: "Apple", 789019: "Microsoft", 1326801: "Meta", 1018724: "Amazon"}
allfin = tc.annual_financials(range(2011, 2026))
for c in ["revenue", "op_income", "net_income", "assets"]:
    allfin[c] = pd.to_numeric(allfin[c], errors="coerce").astype(float)
allfin["sic"] = allfin.cik.map(sic)
air = allfin[allfin.sic == "4512"]
bt = allfin[allfin.cik.isin(BIGTECH)]
print("Airlines found:", ", ".join(sorted(air.groupby("cik").entityName.last().str.title().unique())[:15]))
print("Big tech found:", ", ".join(sorted(bt.groupby("cik").entityName.last().unique())))
agg = lambda d: (d.groupby("year").op_income.sum() / d.groupby("year").revenue.sum() * 100)
cmpm = pd.DataFrame({"airlines_op_margin_pct": agg(air), "bigtech_op_margin_pct": agg(bt),
                     "airlines_op_income_bn": air.groupby("year").op_income.sum() / 1e9,
                     "bigtech_op_income_bn": bt.groupby("year").op_income.sum() / 1e9,
                     "n_airlines": air.groupby("year").size()})
for nm in sorted(set(BIGTECH.values())):
    d = bt[bt.cik.map(BIGTECH) == nm]; cmpm[f"{nm}_op_margin_pct"] = (d.groupby("year").op_income.sum() / d.groupby("year").revenue.sum() * 100)
tc.save_table(cmpm, "h2c_airlines_vs_bigtech")
cmpm.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(cmpm.index, cmpm.bigtech_op_margin_pct, color=tc.C[0], marker="o", ms=5, lw=2, label="Big tech (5 firms, aggregate)")
ax.plot(cmpm.index, cmpm.airlines_op_margin_pct, color=tc.C[1], marker="o", ms=5, lw=2, label="US scheduled airlines (SIC 4512, aggregate)")
ax.axhline(0, color=tc.INK, lw=0.6); ax.axvline(2014.7, color=tc.GREY, lw=0.9)
ax.text(2014.8, ax.get_ylim()[1], " book", fontsize=8, color=tc.MUTED, va="top")
ax.set_ylabel("Operating margin, %"); ax.set_title("Thiel's airline-versus-Google comparison, extended to 2025", loc="left")
ax.legend(loc="lower left"); fig.tight_layout(); tc.save_fig(fig, "h2_fig3_airlines_bigtech")

## Verdict inputs and check cell — send me this output

In [ ]:
def pick(o, sp, x):
    r = reg[(reg.outcome == o) & (reg.spec == sp) & (reg.x == x)]
    return None if r.empty else {k: round(float(r.iloc[0][k]), 4) for k in ["coef", "se", "p"]}

def _num(c):
    try:
        return pd.to_numeric(c)
    except (ValueError, TypeError):
        return c
rnd = lambda df, d=4: df.apply(_num).round(d)
bins_tab = fin.groupby("share_bin", observed=True).agg(n=("margin_dm", "size"), mean_margin=("op_margin", "mean"),
                                                       median_margin_vs_industry=("margin_dm", "median"))
summary = {
    "census_cross_section": rnd(cs).to_dict("records"),
    "census_first_diff": rnd(fdr).to_dict("records"),
    "n_firm_years": int(len(fin)), "n_firms": int(fin.cik.nunique()),
    "opm_share": pick("op_margin", "share", "share"),
    "opm_leader": pick("op_margin", "leader", "leader"),
    "opm_dominant": pick("op_margin", "dominant (>=50%)", "dominant"),
    "opm_share_pre": pick("op_margin", "share, pre vs post 2014", "share"),
    "opm_share_post_interaction": pick("op_margin", "share, pre vs post 2014", "share_x_post"),
    "opm_share_given_size_sector": pick("op_margin", "share + size, sector-year", "share"),
    "opm_size_given_share_sector": pick("op_margin", "share + size, sector-year", "log_rev"),
    "roa_share": pick("roa", "share", "share"),
    "roa_leader": pick("roa", "leader", "leader"),
    "roa_share_given_size_sector": pick("roa", "share + size, sector-year", "share"),
    "share_bins": rnd(bins_tab, 3).reset_index().astype({"share_bin": str}).to_dict("records"),
    "airlines_vs_bigtech_margin": cmpm[["airlines_op_margin_pct", "bigtech_op_margin_pct"]].astype(float).round(1).to_dict("index"),
}
(tc.OUT / "h2_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps(summary, indent=1, default=str))